**Curso superior de Tecnologia em Banco de dados**

**Autor:** Lucas da Silva


**Link do Github:** https://github.com/Lucas-d-Silva/Projeto_steam_games




**Disciplina:** Programção para dados




**Finalidade:** Criar um programa para analisar e responder as perguntas sobre os dados do dataset: Steam_games.csv



**Data:** Agosto de 2026



In [16]:
%%writefile steam/excecoes.py
"""
Módulo de exceções customizadas para a biblioteca 'steam'.
Define uma hierarquia de erros para facilitar o tratamento e diagnóstico no leitor e analisador.
"""
class SteamError(Exception):
    """
    Exceção base para todas as falhas relacionadas ao pacote 'steam'.
    Permite capturar qualquer erro genérico do módulo com um único 'except SteamError'.
    """
    pass

class ArquivoJogosNaoEncontradoError(SteamError):
    """
    Lançada quando o caminho especificado para o arquivo CSV de jogos não existe no sistema de arquivos.
    """
    pass

class FormatoInvalidoError(SteamError):
    """
    Lançada quando o arquivo fornecido não possui a extensão ou estrutura esperada (ex: extensão diferente de .csv).
    """
    pass

class ColunaAusenteError(SteamError):

    """
    Lançada durante a validação do cabeçalho do CSV quando alguma das colunas obrigatórias está ausente.
    """
    pass

Overwriting steam/excecoes.py


In [17]:
%%writefile steam/leitor.py
"""
Módulo responsável pelo carregamento e validação dos dados brutos em arquivo CSV.
Garante a integridade do arquivo antes de disponibilizar os registros para o analisador.
"""
import csv
import os
from .excecoes import ArquivoJogosNaoEncontradoError, FormatoInvalidoError, ColunaAusenteError

class LeitorCSV:
    """
    Classe utilitária para leitura, validação de extensão e verificação de esquemas em arquivos CSV.
    """
    def __init__(self, caminho_arquivo: str):
        """
        Recebe o caminho relativo ou absoluto do arquivo a ser lido.
        """
        self.caminho_arquivo = caminho_arquivo

    def ler_dados(self) -> list[dict]:
        """
        Executa as validações de pré-requisitos e carrega os dados em formato de lista de dicionários.
        Lança exceções customizadas em caso de arquivo inexistente, extensão inválida ou colunas ausentes.
        """

        # 1. Valida se o arquivo realmente existe no disco
        if not os.path.exists(self.caminho_arquivo):
            raise ArquivoJogosNaoEncontradoError(f"Arquivo '{self.caminho_arquivo}' não foi encontrado.")

        # 2. Valida o formato/extensão do arquivo fornecido
        if not self.caminho_arquivo.lower().endswith('.csv'):
            raise FormatoInvalidoError(f"O arquivo '{self.caminho_arquivo}' deve ter a extensão .csv.")

        try:
            # 3. Abre o arquivo com enconding UTF-8 e converte para dicionário
            with open(self.caminho_arquivo, mode='r', encoding='utf-8') as f:
                leitor = csv.DictReader(f)
                dados = list(leitor)

            # Retorna lista vazia caso o CSV esteja completamente em branco
            if not dados:
                return []
            # 4. Validação do schema: garante que todas as colunas essenciais estão presentes
            colunas_obrigatorias = {'Name', 'Price', 'Release date', 'Windows', 'Mac', 'Linux', 'Positive', 'Negative'}
            colunas_presentes = set(dados[0].keys())

            # Verifica se o conjunto obrigatório está contido no cabeçalho do arquivo
            if not colunas_obrigatorias.issubset(colunas_presentes):
                faltantes = colunas_obrigatorias - colunas_presentes
                raise ColunaAusenteError(f"Colunas obrigatórias ausentes: {faltantes}")

            return dados

        except Exception as e:
            # Propaga exceções de domínio já tratadas para manter a precisão do erro
            if isinstance(e, (ArquivoJogosNaoEncontradoError, FormatoInvalidoError, ColunaAusenteError)):
                raise e
                # Encapsula quaisquer outros erros inesperados (ex: IO/Permissão) na exceção base do pacote
            raise SteamError(f"Erro ao ler arquivo CSV: {e}")


Overwriting steam/leitor.py


In [18]:
%%writefile steam/analisador.py
"""
Módulo responsável pelo processamento de dados e regras de negócio da Steam.
Realiza cálculos estatísticos referentes a precificação, datas de lançamento e engajamento.
"""
from datetime import datetime
from .leitor import LeitorCSV

class Analisador:
    """
    Classe principal para análise dos dados da plataforma Steam.
    """
    def __init__(self, caminho_arquivo: str):
        """
        Inicializa o analisador delegando a leitura do CSV ao LeitorCSV.
        """
        leitor = LeitorCSV(caminho_arquivo)
        self.dados = leitor.ler_dados()

    def calcular_percentual_gratuitos_e_pagos(self) -> dict:
        """
        Pergunta 1: Calcula a proporção entre jogos gratuitos (Preço == 0) e pagos.
        Retorna um dicionário com os percentuais arredondados em 2 casas decimais.
        """
        total = len(self.dados)
        if total == 0:
            return {"gratuitos": 0.0, "pagos": 0.0}

        gratuitos = 0
        for jogo in self.dados:
            try:
                # Converte o valor do preço para float e verifica se é zero
                preco = float(jogo.get("Price", 0))
                if preco == 0.0:
                    gratuitos += 1
            except (ValueError, TypeError):
                # Caso haja dado inconsistente ou incorrompido, ignora a linha sem quebrar o fluxo
                continue
        # Cálculo das porcentagens relativas ao total de registros
        perc_gratuitos = round((gratuitos / total) * 100, 2)
        perc_pagos = round(100.0 - perc_gratuitos, 2)

        return {"gratuitos": perc_gratuitos, "pagos": perc_pagos}

    def obter_ano_com_mais_lancamentos(self) -> list[int]:
        """
        Pergunta 2: Identifica o(s) ano(s) com o maior volume de lançamentos na plataforma.
        Trata múltiplos formatos de datas e lida com eventuais empates, retornando uma lista ordenada.
        """
        contagem_anos = {}

        for jogo in self.dados:
            data_str = jogo.get("Release date", "").strip()
            if not data_str:
                continue

            ano = None
            # Tenta fazer o parse da string de data usando diferentes padrões suportados
            for fmt in ("%b %d, %Y", "%b %Y", "%Y"):
                try:
                    ano = datetime.strptime(data_str, fmt).year
                    break # Para no primeiro formato correspondente
                except ValueError:
                    pass
            # Acumula a frequência do ano no dicionário de contagem
            if ano:
                contagem_anos[ano] = contagem_anos.get(ano, 0) + 1

        if not contagem_anos:
            return []
        # Descobre a quantidade máxima de lançamentos e filtra os anos que atingiram essa marca (trata empates)
        max_lancamentos = max(contagem_anos.values())
        anos_mais_lancamentos = [
            ano for ano, count in contagem_anos.items() if count == max_lancamentos
        ]

        return sorted(anos_mais_lancamentos)

    def comparar_engajamento_por_suporte_plataformas(self) -> dict:
        """
        Pergunta 3: Compara a taxa de aprovação média (% de avaliações positivas)
        entre jogos que suportam até 2 sistemas operacionais vs. jogos com suporte total (3 sistemas: Win, Mac, Linux).
        """
        taxas_ate_2 = []
        taxas_3 = []

        for jogo in self.dados:
            # Converte as colunas de suporte para booleano tratável
            win = str(jogo.get("Windows", "False")).strip().lower() in ["true", "1"]
            mac = str(jogo.get("Mac", "False")).strip().lower() in ["true", "1"]
            lin = str(jogo.get("Linux", "False")).strip().lower() in ["true", "1"]
            # Soma a quantidade de plataformas suportadas (0 a 3)
            qtd_sistemas = sum([win, mac, lin])

            try:
                pos = float(jogo.get("Positive", 0))
                neg = float(jogo.get("Negative", 0))
            except (ValueError, TypeError):
                continue

            total_votos = pos + neg
            # Evita divisão por zero para jogos sem nenhuma avaliação cadastrada
            if total_votos == 0:
                continue
            # Calcula a taxa individual de aprovação do jogo em porcentagem
            taxa = (pos / total_votos) * 100
            # Agrupa os resultados conforme a quantidade de sistemas operacionais suportados
            if qtd_sistemas <= 2:
                taxas_ate_2.append(taxa)
            elif qtd_sistemas == 3:
                taxas_3.append(taxa)
        # Calcula a média aritmética de aprovação para cada grupo
        med_2 = sum(taxas_ate_2) / len(taxas_ate_2) if taxas_ate_2 else 0.0
        med_3 = sum(taxas_3) / len(taxas_3) if taxas_3 else 0.0

        dif = round(med_3 - med_2, 2)

        return {
            "total_ate_2_sistemas": len(taxas_ate_2),
            "taxa_aprovacao_ate_2": round(med_2, 2),
            "total_3_sistemas": len(taxas_3),
            "taxa_aprovacao_todos_3": round(med_3, 2),
            "diferenca_pontos_percentuais": dif
        }



Overwriting steam/analisador.py


In [19]:
%%writefile steam/__init__.py
"""
Módulo de inicialização do pacote 'steam'.
Este arquivo expõe as principais classes e exceções customizadas para simplificar
as importações externas (interface pública do pacote).
"""

#Importa a classe responsável pelo carregamento e validação dos dados em CSV
from .leitor import LeitorCSV
#Importa a hierarquia de exceções customizadas para tratamento de erros
from .excecoes import (
    SteamError,
    ArquivoJogosNaoEncontradoError,
    FormatoInvalidoError,
    ColunaAusenteError
)
# Importa a classe responsável pelas regras de negócio e análises estatísticas
from .analisador import Analisador
#Define explicitamente quais símbolos são exportados quando alguém utiliza 'from steam import *'
__all__ = [
    'LeitorCSV',
    'SteamError',
    'ArquivoJogosNaoEncontradoError',
    'FormatoInvalidoError',
    'ColunaAusenteError',
    'Analisador'
]


Overwriting steam/__init__.py


In [20]:
%%writefile test_sistema.py
"""
Script de testes automatizados para validação do sistema 'steam'.
Verifica a acurácia dos cálculos com base na amostra de controle e atesta o disparo correto de exceções.
"""
import os
from steam import Analisador, ArquivoJogosNaoEncontradoError

def rodar_testes():
    """
    Executa a suíte de testes do sistema e compara os resultados com os gabaritos esperados.
    """
    print("=== INICIANDO TESTES DO SISTEMA STEAM (AMOSTRA DE 20 JOGOS) ===")

    # 1. Validação de pré-requisito: verifica existência da amostra no ambiente
    caminho_amostra = "amostra_20.csv"
    if not os.path.exists(caminho_amostra):
        print(f"❌ Erro: Arquivo '{caminho_amostra}' não encontrado. Faça o upload do CSV.")
        return

    # 2. Teste de Instanciação e Carregamento de Dados
    analisador = Analisador(caminho_amostra)
    assert len(analisador.dados) == 20, f"Esperado 20 jogos, mas carregou {len(analisador.dados)}"
    print(f"\n[1] Carregamento: SUCESSO ({len(analisador.dados)} jogos)")

    # Definção dos valores esperados (Gabarito Oficial para a amostra de 20 registros)
    GABARITO_P1 = {"gratuitos": 5.0, "pagos": 95.0}
    GABARITO_P2 =  [2018]
    GABARITO_P3_DIF = 18.28

    # 3. Teste da Pergunta 1: Percentual de jogos gratuitos e pagos
    p1 = analisador.calcular_percentual_gratuitos_e_pagos()
    assert p1 == GABARITO_P1, f"❌ Falha P1: Calculado {p1} difere de {GABARITO_P1}"
    print(f"[2] Pergunta 1: APROVADO ✅ ({p1})")

    # 4. Teste da Pergunta 2: Ano com mais lançamentos
    p2 = sorted(analisador.obter_ano_com_mais_lancamentos())
    assert p2 == GABARITO_P2, f"❌ Falha P2: Calculado {p2} difere de {GABARITO_P2}"
    print(f"[3] Pergunta 2: APROVADO ✅ (Anos: {p2})")

    # 5. Teste da Pergunta 3: Comparativo de engajamento por suporte a plataformas
    p3 = analisador.comparar_engajamento_por_suporte_plataformas()
    dif = p3.get("diferenca_pontos_percentuais")
    assert dif == GABARITO_P3_DIF, f"❌ Falha P3: Calculado {dif} difere de {GABARITO_P3_DIF}"
    print(f"[4] Pergunta 3: APROVADO ✅ (Diferença: {dif} p.p.)")

    # 6. Teste de Tratamento de Exceções: Garante disparo da exceção correta para arquivo inexistente
    print("\n[5] Teste de Exceção (Arquivo Inexistente):")
    try:
        Analisador("arquivo_inexistente.csv")
        print("❌ Falha: Deveria ter lançado a exceção.")
    except ArquivoJogosNaoEncontradoError as e:
        print(f"    - Sucesso! Exceção capturada corretamente: {e}")

    print("\n🎉 TODOS OS TESTES PASSARAM E FORAM VALIDADOS COM SUCESSO!")

# Ponto de entrada do script ao ser executado via terminal ou subprocesso
if __name__ == "__main__":
    rodar_testes()


Overwriting test_sistema.py


In [1]:
import csv
import random

# Define uma semente fixa para que os testes sejam reproduzíveis
random.seed(42)

caminho_entrada = "steam_games.csv"
caminho_saida = "amostra_20.csv"

with open(caminho_entrada, mode="r", encoding="utf-8") as file_in:
    leitor = list(csv.reader(file_in))
    cabecalho = leitor[0]
    linhas_dados = leitor[1:]

# Seleciona 20 jogos aleatórios da base completa
amostra_aleatoria = random.sample(linhas_dados, 20)

# Salva a amostra sorteada no arquivo amostra_20.csv
with open(caminho_saida, mode="w", encoding="utf-8", newline="") as file_out:
    escritor = csv.writer(file_out)
    escritor.writerow(cabecalho)
    escritor.writerows(amostra_aleatoria)

print("Amostra de 20 jogos gerada com sucesso em 'amostra_20.csv'!")

Amostra de 20 jogos gerada com sucesso em 'amostra_20.csv'!


In [2]:
import importlib
import sys

sys.path.append(".")

# Recarrega e importa o pacote principal
import steam.analisador

importlib.reload(steam.analisador)

from steam.analisador import Analisador
from steam.excecoes import ArquivoJogosNaoEncontradoError, FormatoInvalidoError
from steam.leitor import LeitorCSV

print("✅ Módulos prontos e atualizados!")

✅ Módulos prontos e atualizados!


In [9]:
from steam import Analisador

# Instancia o analisador com o seu arquivo de dados
analisador = Analisador("steam_games.csv")


print("ANALISE DAS PERGUNTAS DE STEAM_GAMES")

# --- Pergunta 1 ---
p1 = analisador.calcular_percentual_gratuitos_e_pagos()
print("\n1. PROPORÇÃO DE JOGOS GRATUITOS E PAGOS:")
print(f"   • Jogos Gratuitos: {p1['gratuitos']}%")
print(f"   • Jogos Pagos:     {p1['pagos']}%")

# --- Pergunta 2 ---
p2 = analisador.obter_ano_com_mais_lancamentos()
anos_str = ", ".join(map(str, p2))
print("\n2. PICO DE LANÇAMENTOS NA PLATAFORMA:")
print(f"   • Ano(s) com maior volume de lançamentos: {anos_str}")

# --- Pergunta 3 ---
p3 = analisador.comparar_engajamento_por_suporte_plataformas()

print("\n3. ENGAJAMENTO POR SUPORTE DE PLATAFORMAS:")
print(f"   • Até 2 Sistemas {p3['taxa_aprovacao_ate_2']}% de aprovação ({p3['total_ate_2_sistemas']} jogos)")
print(f"   • Todos os 3 Sistemas (Win/Mac/Linux): {p3['taxa_aprovacao_todos_3']}% de aprovação ({p3['total_3_sistemas']} jogos)")
print(f"   • Diferença de Engajamento:      {p3['diferenca_pontos_percentuais']} p.p.")

print("\n" + "=" * 60)

ANALISE DAS PERGUNTAS DE STEAM_GAMES

1. PROPORÇÃO DE JOGOS GRATUITOS E PAGOS:
   • Jogos Gratuitos: 17.39%
   • Jogos Pagos:     82.61%

2. PICO DE LANÇAMENTOS NA PLATAFORMA:
   • Ano(s) com maior volume de lançamentos: 2022

3. ENGAJAMENTO POR SUPORTE DE PLATAFORMAS:
   • Até 2 Sistemas 74.06% de aprovação (51071 jogos)
   • Todos os 3 Sistemas (Win/Mac/Linux): 78.72% de aprovação (7059 jogos)
   • Diferença de Engajamento:      4.66 p.p.



In [13]:
import importlib
import test_sistema

# Recarrega o módulo test_sistema para garantir que as últimas alterações sejam aplicadas
importlib.reload(test_sistema)

# Corrige a importação da classe renomeada
from steam import Analisador

# Executa a suíte de testes automatizados
test_sistema.rodar_testes()

=== INICIANDO TESTES DO SISTEMA STEAM (AMOSTRA DE 20 JOGOS) ===

[1] Carregamento: SUCESSO (20 jogos)
[2] Pergunta 1: APROVADO ✅ ({'gratuitos': 5.0, 'pagos': 95.0})
[3] Pergunta 2: APROVADO ✅ (Anos: [2018])
[4] Pergunta 3: APROVADO ✅ (Diferença: 18.28 p.p.)

[5] Teste de Exceção (Arquivo Inexistente):
    - Sucesso! Exceção capturada corretamente: Arquivo 'arquivo_inexistente.csv' não foi encontrado.

🎉 TODOS OS TESTES PASSARAM E FORAM VALIDADOS COM SUCESSO!
